# Fidelity human-review analysis

**Inputs**
- `agent/logs/v2/websiteratings.csv` — 16 reviewers × 40 stimuli × 5 fields
- `scripts/fidelity_check/screenshots/rating_manifest.json` — ground-truth stim → source (rng_seed=42, 20 ours + 20 real)

**Outputs**
- `agent/logs/v2/app_fidelity.tex` — LaTeX snippet `\input`'d from the paper appendix `app:fidelity`
- Numbers cited in the main paper §3 fidelity paragraph

**Reviewers.** $n=16$, all computer-science background, ages 21–40, self-reported phishing-security familiarity Low / Medium / High.

**Stimuli.** 40 interleaved screenshots = 20 Scammer4U environments + 20 real captured-phishing pages (PhishTank / Wayback / curated). Per-item rating: three 1–5 Likerts (visual believability, copy quality, would-fool-a-user), 3-class source guess (Ours / Real / Not sure), and confidence (Low / Medium / High).

**Analyses** (per `paper-plan.md` §1.4 + `analysis-plan.md` §11 D11):
1. Source-discrimination accuracy — three Not-Sure framings (strict / exclude / 3-class)
2. Per-axis Likert: ours vs real (Visual / Copy / Would-Fool), Mann–Whitney $U$
3. Stratification by reviewer familiarity
4. Confidence sub-stratification (High-confidence subset)
5. Not-sure rate by source
6. Inter-rater agreement (Fleiss $\kappa$ on source guess, Krippendorff $\alpha$-interval on Likerts)
7. Per-stim breakdown (which envs got fooled most)
8. LaTeX dump

Outside `analysis-plan.md` §6 BH family — descriptive context for paper §3.

In [24]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

# Walk up from notebook cwd until we find CLAUDE.md (repo root marker)
HERE = Path.cwd()
ROOT = HERE
while ROOT != ROOT.parent and not (ROOT / 'CLAUDE.md').exists():
    ROOT = ROOT.parent
assert (ROOT / 'CLAUDE.md').exists(), f'Could not find repo root from {HERE}'

CSV_PATH      = ROOT / 'agent' / 'logs' / 'v2' / 'websiteratings.csv'
MANIFEST_PATH = ROOT / 'scripts' / 'fidelity_check' / 'screenshots' / 'rating_manifest.json'
LATEX_OUT     = ROOT / 'agent' / 'logs' / 'v2' / 'app_fidelity.tex'

print('ROOT     :', ROOT)
print('CSV      :', CSV_PATH, '— exists:', CSV_PATH.exists())
print('manifest :', MANIFEST_PATH, '— exists:', MANIFEST_PATH.exists())
print('out tex  :', LATEX_OUT)

ROOT     : c:\Users\Soham\Documents\NGMI26\Scammer4U
CSV      : c:\Users\Soham\Documents\NGMI26\Scammer4U\agent\logs\v2\websiteratings.csv — exists: True
manifest : c:\Users\Soham\Documents\NGMI26\Scammer4U\scripts\fidelity_check\screenshots\rating_manifest.json — exists: True
out tex  : c:\Users\Soham\Documents\NGMI26\Scammer4U\agent\logs\v2\app_fidelity.tex


In [25]:
raw = pd.read_csv(CSV_PATH)
print(f'Raw shape: {raw.shape}  (expected 16 rows + ~203 cols)')
fam_col = [c for c in raw.columns if 'familiarity' in c.lower()][0]
print(f'Familiarity column: {fam_col!r}')

with open(MANIFEST_PATH) as f:
    manifest = json.load(f)
gt = {item['number']: item for item in manifest['items']}
n_ours = sum(1 for v in gt.values() if v['source'] == 'ours')
n_real = sum(1 for v in gt.values() if v['source'] == 'real')
print(f'Manifest: {len(gt)} stimuli  ({n_ours} ours, {n_real} real)')

Raw shape: (16, 203)  (expected 16 rows + ~203 cols)
Familiarity column: 'How would you describe your familiarity with phishing / web security?'
Manifest: 40 stimuli  (20 ours, 20 real)


## Reshape wide → long

640 rows (16 reviewers × 40 stimuli). Each row: user, familiarity, stim, ratings, guess, confidence, ground-truth source.

In [26]:
long_rows = []
for _, row in raw.iterrows():
    user = row['Username']
    fam = row[fam_col]
    for n in range(1, 41):
        p = f'stim_{n:02d}'
        long_rows.append({
            'user': user,
            'familiarity': fam,
            'stim': n,
            'visual':     row.get(f'{p} — Visual Believability'),
            'copy':       row.get(f'{p} — Copy Quality'),
            'fool':       row.get(f'{p} — Would Fool a User'),
            'guess':      row.get(f'{p} — Source'),
            'confidence': row.get(f'{p} — Confidence'),
            'gt_source':  gt[n]['source'],
            'gt_name':    gt[n]['original_name'],
        })
df = pd.DataFrame(long_rows)
for col in ['visual', 'copy', 'fool']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

def normalize_guess(g):
    if pd.isna(g):
        return 'missing'
    s = str(g).lower()
    if 'ours' in s or 'synth' in s:   return 'ours'
    if 'real' in s or 'phish' in s:   return 'real'
    if 'not sure' in s or 'unsure' in s: return 'not_sure'
    return 'missing'

df['guess_norm'] = df['guess'].apply(normalize_guess)
df['correct_strict'] = (df['guess_norm'] == df['gt_source']).astype(int)

print(f'Long shape: {df.shape}  (expected 640 rows)')
print('Guess distribution:')
print(df['guess_norm'].value_counts())
print('\nReviewers by familiarity:')
print(raw[fam_col].value_counts())

Long shape: (640, 12)  (expected 640 rows)
Guess distribution:
guess_norm
real        311
ours        252
not_sure     77
Name: count, dtype: int64

Reviewers by familiarity:
How would you describe your familiarity with phishing / web security?
Low       7
Medium    6
High      3
Name: count, dtype: int64


In [27]:
def wilson_ci(k, n, z=1.96):
    if n == 0:
        return (np.nan, np.nan)
    p = k / n
    denom = 1 + z*z/n
    center = (p + z*z/(2*n)) / denom
    halfw = z * np.sqrt(p*(1-p)/n + z*z/(4*n*n)) / denom
    return (center - halfw, center + halfw)

def fmt_pct_ci(k, n):
    if n == 0:
        return 'n/a'
    lo, hi = wilson_ci(k, n)
    return f'{100*k/n:5.1f}%  [{100*lo:.1f}, {100*hi:.1f}]   n={n}'

def sig_vs_chance(k, n, p0=0.5):
    """Two-sided exact binomial test vs chance line p0."""
    if n == 0:
        return np.nan
    return stats.binomtest(k, n, p=p0, alternative='two-sided').pvalue

## 1. Source-discrimination accuracy — three Not-Sure framings

- **A. Strict.** Not Sure → wrong. Chance line = 50%.
- **B. Exclude Not Sure.** Only committed guesses. Chance line = 50%.
- **C. Three-class.** Report `correct / wrong-commit / not-sure` rates separately. No single chance line.

In [28]:
# A: strict
k_strict = df['correct_strict'].sum()
n_strict = (df['guess_norm'] != 'missing').sum()
p_strict = sig_vs_chance(k_strict, n_strict, 0.5)

# B: exclude not_sure
b_df = df[df['guess_norm'].isin(['ours', 'real'])]
k_b = (b_df['guess_norm'] == b_df['gt_source']).sum()
n_b = len(b_df)
p_b = sig_vs_chance(k_b, n_b, 0.5)

# C: three-way
n_total = (df['guess_norm'] != 'missing').sum()
n_correct = df['correct_strict'].sum()
n_not_sure = (df['guess_norm'] == 'not_sure').sum()
n_wrong = n_total - n_correct - n_not_sure

print('=' * 64)
print('HEADLINE — source-discrimination accuracy')
print('=' * 64)
print(f'\nA. Strict (Not Sure = wrong)        : {fmt_pct_ci(k_strict, n_strict)}  p(vs 50%)={p_strict:.4f}')
print(f'B. Exclude Not Sure                  : {fmt_pct_ci(k_b, n_b)}  p(vs 50%)={p_b:.4f}')
print(f'C. Three-class breakdown (n={n_total}):')
print(f'     correct       : {100*n_correct/n_total:5.1f}%  ({n_correct})')
print(f'     wrong-commit  : {100*n_wrong/n_total:5.1f}%  ({n_wrong})')
print(f'     not sure      : {100*n_not_sure/n_total:5.1f}%  ({n_not_sure})')

# By ground truth source
print('\n' + '-' * 64)
print('By ground-truth source (excl. Not Sure)')
print('-' * 64)
by_src = {}
for src in ['ours', 'real']:
    sub = b_df[b_df['gt_source'] == src]
    k = (sub['guess_norm'] == src).sum()
    n = len(sub)
    by_src[src] = (k, n)
    print(f'  gt={src:>4s} caught correctly: {fmt_pct_ci(k, n)}')

# 3-way crosstab
print('\nFull crosstab (rows = ground truth, cols = guess):')
ct = pd.crosstab(df['gt_source'], df['guess_norm'], normalize='index') * 100
print(ct.round(1).to_string())

HEADLINE — source-discrimination accuracy

A. Strict (Not Sure = wrong)        :  47.5%  [43.7, 51.4]   n=640  p(vs 50%)=0.2204
B. Exclude Not Sure                  :  54.0%  [49.9, 58.1]   n=563  p(vs 50%)=0.0636
C. Three-class breakdown (n=640):
     correct       :  47.5%  (304)
     wrong-commit  :  40.5%  (259)
     not sure      :  12.0%  (77)

----------------------------------------------------------------
By ground-truth source (excl. Not Sure)
----------------------------------------------------------------
  gt=ours caught correctly:  48.8%  [43.0, 54.6]   n=283
  gt=real caught correctly:  59.3%  [53.4, 64.9]   n=280

Full crosstab (rows = ground truth, cols = guess):
guess_norm  not_sure  ours  real
gt_source                       
ours            11.6  43.1  45.3
real            12.5  35.6  51.9


## 2. Per-axis Likert: ours vs real

Mann–Whitney $U$ on each of Visual / Copy / Would-Fool. If ours $\approx$ real on Visual + Copy and ours $\geq$ real on Would-Fool, the templated envs are indistinguishable from real captures on the dimensions reviewers were asked to weigh.

In [29]:
likert_stats = {}
print('=' * 64)
print('Per-axis Likert: ours vs real')
print('=' * 64)
for axis in ['visual', 'copy', 'fool']:
    o = df[df['gt_source'] == 'ours'][axis].dropna()
    r = df[df['gt_source'] == 'real'][axis].dropna()
    u, p = stats.mannwhitneyu(o, r, alternative='two-sided')
    o_mean, o_sd = o.mean(), o.std()
    r_mean, r_sd = r.mean(), r.std()
    o_se = o_sd / np.sqrt(len(o))
    r_se = r_sd / np.sqrt(len(r))
    likert_stats[axis] = dict(o_mean=o_mean, o_sd=o_sd, o_se=o_se, o_n=len(o),
                              r_mean=r_mean, r_sd=r_sd, r_se=r_se, r_n=len(r),
                              u=u, p=p, delta=o_mean - r_mean)
    print(f'\n{axis:>6s}: ours {o_mean:.2f} ± {o_sd:.2f}  (median {o.median():.0f}, n={len(o)})')
    print(f'         real {r_mean:.2f} ± {r_sd:.2f}  (median {r.median():.0f}, n={len(r)})')
    print(f'         Δ(ours-real) = {o_mean - r_mean:+.2f}   Mann-Whitney U={u:.0f}, p={p:.4f}')

Per-axis Likert: ours vs real

visual: ours 3.45 ± 1.28  (median 4, n=320)
         real 2.80 ± 1.53  (median 3, n=320)
         Δ(ours-real) = +0.65   Mann-Whitney U=63602, p=0.0000

  copy: ours 3.45 ± 1.23  (median 4, n=320)
         real 2.92 ± 1.57  (median 3, n=320)
         Δ(ours-real) = +0.53   Mann-Whitney U=60800, p=0.0000

  fool: ours 3.51 ± 1.17  (median 4, n=320)
         real 2.86 ± 1.47  (median 3, n=320)
         Δ(ours-real) = +0.65   Mann-Whitney U=63836, p=0.0000


## 3. Stratification by reviewer familiarity

Low / Medium / High self-report. The High tier is the toughest test (smallest cell; caveat the $n$ explicitly).

In [30]:
fam_stats = {}
print('=' * 64)
print('Accuracy by reviewer familiarity (excl. Not Sure)')
print('=' * 64)
for tier in ['Low', 'Medium', 'High']:
    sub = b_df[b_df['familiarity'] == tier]
    k = (sub['guess_norm'] == sub['gt_source']).sum()
    n = len(sub)
    n_rev = sub['user'].nunique()
    p_chance = sig_vs_chance(k, n, 0.5)
    fam_stats[tier] = dict(k=k, n=n, n_rev=n_rev, p=p_chance)
    print(f'  {tier:>6s} (n_reviewers={n_rev}): {fmt_pct_ci(k, n)}  p(vs 50%)={p_chance:.4f}')

Accuracy by reviewer familiarity (excl. Not Sure)
     Low (n_reviewers=7):  56.3%  [50.0, 62.3]   n=247  p(vs 50%)=0.0561
  Medium (n_reviewers=6):  53.7%  [47.0, 60.2]   n=216  p(vs 50%)=0.3074
    High (n_reviewers=3):  49.0%  [39.4, 58.7]   n=100  p(vs 50%)=0.9204


## 4. Confidence sub-stratification

Restrict to ratings the reviewer marked High-confidence. If they're at chance even when they say they're sure, the discrimination really is hard.

In [31]:
conf_stats = {}
print('=' * 64)
print('Accuracy by reviewer confidence (excl. Not Sure)')
print('=' * 64)
for c in ['Low', 'Medium', 'High']:
    sub = b_df[b_df['confidence'] == c]
    k = (sub['guess_norm'] == sub['gt_source']).sum()
    n = len(sub)
    p_chance = sig_vs_chance(k, n, 0.5)
    conf_stats[c] = dict(k=k, n=n, p=p_chance)
    print(f'  {c:>6s}-conf: {fmt_pct_ci(k, n)}  p(vs 50%)={p_chance:.4f}')

print('\nConfidence × correctness crosstab (counts):')
print(pd.crosstab(df['confidence'], df['correct_strict'], margins=True).to_string())

Accuracy by reviewer confidence (excl. Not Sure)
     Low-conf:  50.0%  [33.2, 66.8]   n=30  p(vs 50%)=1.0000
  Medium-conf:  54.4%  [47.5, 61.0]   n=206  p(vs 50%)=0.2362
    High-conf:  54.1%  [48.7, 59.5]   n=327  p(vs 50%)=0.1504

Confidence × correctness crosstab (counts):
correct_strict    0    1  All
confidence                   
High            177  177  354
Low              31   15   46
Medium          128  112  240
All             336  304  640


## 5. Not-sure rate by source

High not-sure rate on ours is itself evidence of plausibility (reviewers couldn't commit).

In [32]:
ns_stats = {}
print('=' * 64)
print('Not-sure rate by ground-truth source')
print('=' * 64)
for src in ['ours', 'real']:
    sub = df[df['gt_source'] == src]
    ns = (sub['guess_norm'] == 'not_sure').sum()
    n = len(sub)
    ns_stats[src] = dict(ns=ns, n=n, rate=ns/n)
    print(f'  gt={src}: {100*ns/n:.1f}% Not Sure  ({ns}/{n})')
ns_overall = (df['guess_norm'] == 'not_sure').sum()
ns_stats['overall'] = dict(ns=ns_overall, n=len(df), rate=ns_overall/len(df))
print(f'  overall: {100*ns_overall/len(df):.1f}%  ({ns_overall}/{len(df)})')

Not-sure rate by ground-truth source
  gt=ours: 11.6% Not Sure  (37/320)
  gt=real: 12.5% Not Sure  (40/320)
  overall: 12.0%  (77/640)


## 6. Inter-rater agreement

- **Source guess** (3 categorical): Fleiss $\kappa$.
- **Likerts** (1–5 ordinal): Krippendorff $\alpha$-interval (treats Likert as interval; standard approximation for 5-point scales; tightest defensible without rank-distance machinery).

In [33]:
def fleiss_kappa(M):
    """M: items x categories matrix of rating counts. Fixed n_raters per item."""
    n_items, n_cat = M.shape
    n_raters = M[0].sum()
    p_j = M.sum(axis=0) / (n_items * n_raters)
    P_i = (np.sum(M**2, axis=1) - n_raters) / (n_raters * (n_raters - 1))
    P_bar = P_i.mean()
    P_e = (p_j**2).sum()
    return (P_bar - P_e) / (1 - P_e)

cats = ['ours', 'real', 'not_sure']
M = np.zeros((40, 3), dtype=int)
for n in range(1, 41):
    sub = df[df['stim'] == n]
    for j, c in enumerate(cats):
        M[n-1, j] = (sub['guess_norm'] == c).sum()
assert all(M.sum(axis=1) == 16), 'Imbalanced raters per item'
kappa_src = fleiss_kappa(M)
print(f'Fleiss κ (source guess, 3 categories): {kappa_src:.3f}')

def krippendorff_alpha_interval(items):
    """Krippendorff α (interval / squared-difference). items: list of rating lists."""
    obs_sum, obs_cnt = 0.0, 0
    for ratings in items:
        rs = [r for r in ratings if pd.notna(r)]
        for i in range(len(rs)):
            for j in range(len(rs)):
                if i != j:
                    obs_sum += (rs[i] - rs[j]) ** 2
                    obs_cnt += 1
    Do = obs_sum / obs_cnt if obs_cnt else np.nan
    all_r = np.array([r for ratings in items for r in ratings if pd.notna(r)], dtype=float)
    De = 2 * np.var(all_r)
    return np.nan if De == 0 else 1 - Do / De

alpha_stats = {}
for axis in ['visual', 'copy', 'fool']:
    items = [df[df['stim'] == n][axis].tolist() for n in range(1, 41)]
    a = krippendorff_alpha_interval(items)
    alpha_stats[axis] = a
    print(f'Krippendorff α (interval) for {axis}: {a:.3f}')

Fleiss κ (source guess, 3 categories): 0.020
Krippendorff α (interval) for visual: 0.259
Krippendorff α (interval) for copy: 0.245
Krippendorff α (interval) for fool: 0.256


## 7. Per-stim breakdown

Which of our envs got mistaken for real most often (low per-stim accuracy on ours = good for us). Which real captures got mistaken for ours (low per-stim accuracy on real = our reviewers found real phishing as templated-looking as our work).

In [34]:
rows = []
for n in range(1, 41):
    sub = df[df['stim'] == n]
    src = gt[n]['source']
    orig = gt[n]['original_name']
    excl = sub[sub['guess_norm'].isin(['ours', 'real'])]
    correct = (excl['guess_norm'] == src).sum()
    n_comm = len(excl)
    ns = (sub['guess_norm'] == 'not_sure').sum()
    rows.append(dict(
        stim=n, gt=src, name=orig,
        n_correct=correct, n_committed=n_comm, n_not_sure=ns,
        accuracy = correct / n_comm if n_comm else np.nan,
        visual=sub['visual'].mean(), copy=sub['copy'].mean(), fool=sub['fool'].mean(),
    ))
ps = pd.DataFrame(rows)

print('OURS envs (low accuracy = mistaken for real → good for us)')
print('-' * 84)
print(ps[ps['gt'] == 'ours'].sort_values('accuracy')[
    ['stim', 'name', 'n_correct', 'n_committed', 'n_not_sure', 'accuracy', 'visual', 'copy', 'fool']
].to_string(index=False, float_format=lambda x: f'{x:.2f}'))

print('\nREAL envs (low accuracy = mistaken for ours)')
print('-' * 84)
print(ps[ps['gt'] == 'real'].sort_values('accuracy')[
    ['stim', 'name', 'n_correct', 'n_committed', 'n_not_sure', 'accuracy', 'visual', 'copy', 'fool']
].to_string(index=False, float_format=lambda x: f'{x:.2f}'))

print(f'\nSummary: mean per-stim accuracy ours={ps[ps.gt=="ours"].accuracy.mean():.2f}'
      f'real={ps[ps.gt=="real"].accuracy.mean():.2f}')

OURS envs (low accuracy = mistaken for real → good for us)
------------------------------------------------------------------------------------
 stim                name  n_correct  n_committed  n_not_sure  accuracy  visual  copy  fool
   25 e14_oracle_no_timer          3           16           0      0.19    3.94  3.94  3.94
   26          e17_amazon          4           15           1      0.27    3.81  3.81  3.88
    8            e8_chase          3           11           5      0.27    3.88  3.62  3.62
   38    e13_irs_no_timer          5           14           2      0.36    3.50  3.56  3.44
   37         e23_netflix          5           14           2      0.36    3.38  3.12  3.62
   34     crypto_platform          6           15           1      0.40    4.00  4.12  4.12
    3                news          6           15           1      0.40    4.19  4.12  4.06
   32       dating_verify          6           14           2      0.43    3.62  3.38  3.56
    4         e10_mychart   

KeyError: False

## Done

**Where the numbers go in the paper:**
- **Main paper §3 fidelity paragraph** (1 short paragraph, ~6–8 lines): quote Framing-A accuracy with CI + Mann–Whitney $p$ on the Would-Fool axis + sample-and-protocol one-liner; point to `app:fidelity`.
- **Appendix `app:fidelity`** (just-emitted `.tex`): sample/protocol, three-framing accuracy table, per-axis Likert table, stratification by familiarity, High-confidence subset, inter-rater agreement, example-screenshot figure (placeholder — Soham drops the grid in), per-stim longtable.

**Supersedes:** `scripts/fidelity_check/04_analyze_ratings.py` for paper-time numbers (kept on disk for back-compat with earlier README; cross-reference there if needed).

**Outstanding (manual):**
- Drop the example-screenshot grid into `fig/fidelity_examples.pdf` (or analogue) and uncomment the `\includegraphics` line in `app_fidelity.tex` (the `\fbox` placeholder will compile in the meantime).
- `\usepackage{longtable}` in the paper preamble if not already there.
- Update `CLAUDE.md` “Outstanding bookkeeping” / `paper-plan.md` §1.4 to flip the fidelity TODO from “pending” to “landed” once the screenshot grid is in.